# Find Dance major count

In [ ]:
from pathlib import Path
import pandas as pd
import re
import time
import gc

# ============================================================
# FIND DANCE MAJOR COUNT
# READ ONLY
# NO SAVE
# MEMORY OPTIMIZED
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

CHUNK_SIZE = 200_000

print("=" * 100)
print("FINDING DANCE MAJOR COUNT")
print("=" * 100)
print("File:", DEGREE_FILE)

if not DEGREE_FILE.exists():
    raise FileNotFoundError(f"File not found: {DEGREE_FILE}")

start_time = time.time()

# ------------------------------------------------------------
# 1. Read columns only
# ------------------------------------------------------------
header = pd.read_csv(DEGREE_FILE, nrows=0, low_memory=False)
cols = list(header.columns)

def clean_name(x):
    return str(x).strip().lower().replace(" ", "_")

col_map = {clean_name(c): c for c in cols}
clean_cols = list(col_map.keys())

print("\nColumns found:", len(cols))

# ------------------------------------------------------------
# 2. Auto-detect important columns
# ------------------------------------------------------------

# Year column
year_candidates = [
    "year", "academic_year", "survey_year", "release_year",
    "data_year", "report_year"
]

year_col = None
for c in year_candidates:
    if c in col_map:
        year_col = col_map[c]
        break

if year_col is None:
    for c in cols:
        if "year" in clean_name(c):
            year_col = c
            break

# Major / CIP title columns
major_title_cols = []
for c in cols:
    cc = clean_name(c)
    if (
        "cip" in cc
        or "major" in cc
        or "program" in cc
        or "title" in cc
        or "desc" in cc
    ):
        major_title_cols.append(c)

# CIP code columns
cip_code_cols = []
for c in cols:
    cc = clean_name(c)
    if cc in ["cipcode", "cip_code", "cip", "cip_code_6_digit"]:
        cip_code_cols.append(c)
    elif "cip" in cc and "code" in cc:
        cip_code_cols.append(c)

# Award / degree level column
degree_level_col = None
degree_level_candidates = [
    "awlevel", "award_level", "degree_level", "level", "credential_level"
]

for c in degree_level_candidates:
    if c in col_map:
        degree_level_col = col_map[c]
        break

if degree_level_col is None:
    for c in cols:
        cc = clean_name(c)
        if "award" in cc or "degree_level" in cc or cc == "level":
            degree_level_col = c
            break

# Count column
count_col = None

count_priority = [
    "ctotalt",
    "total",
    "total_count",
    "count",
    "degree_count",
    "completions",
    "completion_count",
    "awards",
    "award_count",
    "graduates",
    "graduate_count"
]

for c in count_priority:
    if c in col_map:
        count_col = col_map[c]
        break

if count_col is None:
    for c in cols:
        cc = clean_name(c)
        if (
            "total" in cc
            or "count" in cc
            or "completion" in cc
            or "award" in cc
            or "graduate" in cc
        ):
            count_col = c
            break

print("\nDetected columns:")
print("Year column:", year_col)
print("Degree level column:", degree_level_col)
print("Count column:", count_col)
print("Possible major/title columns:", major_title_cols[:10])
print("Possible CIP code columns:", cip_code_cols)

# ------------------------------------------------------------
# 3. Choose only needed columns
# ------------------------------------------------------------
usecols = []

for c in [year_col, degree_level_col, count_col]:
    if c is not None and c not in usecols:
        usecols.append(c)

for c in major_title_cols + cip_code_cols:
    if c not in usecols:
        usecols.append(c)

print("\nReading only these columns:")
for c in usecols:
    print("-", c)

# ------------------------------------------------------------
# 4. Search Dance majors
# ------------------------------------------------------------
results = []
chunk_num = 0
total_rows_seen = 0
dance_rows_found = 0

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    low_memory=False,
    dtype=str
):
    chunk_num += 1
    total_rows_seen += len(chunk)

    # Build search text from major/title columns
    search_text = pd.Series("", index=chunk.index, dtype="string")

    for c in major_title_cols:
        if c in chunk.columns:
            search_text = search_text + " " + chunk[c].fillna("").astype(str)

    # Dance title match
    dance_title_match = search_text.str.contains(r"\bdance\b", case=False, regex=True, na=False)

    # Dance CIP match
    # Dance CIP usually begins with 50.03
    dance_cip_match = pd.Series(False, index=chunk.index)

    for c in cip_code_cols:
        if c in chunk.columns:
            cip_clean = (
                chunk[c]
                .fillna("")
                .astype(str)
                .str.replace(".", "", regex=False)
                .str.replace("-", "", regex=False)
                .str.strip()
            )

            # Match 50.03xx style after removing punctuation
            dance_cip_match = dance_cip_match | cip_clean.str.startswith("5003")

    dance_match = dance_title_match | dance_cip_match

    dance_chunk = chunk.loc[dance_match].copy()

    if len(dance_chunk) > 0:
        dance_rows_found += len(dance_chunk)

        if count_col is not None:
            dance_chunk[count_col] = (
                dance_chunk[count_col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("*", "", regex=False)
                .str.strip()
            )
            dance_chunk[count_col] = pd.to_numeric(dance_chunk[count_col], errors="coerce").fillna(0)
        else:
            dance_chunk["record_count"] = 1

        results.append(dance_chunk)

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {total_rows_seen:,} | "
            f"dance rows found: {dance_rows_found:,} | total min: {elapsed:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

# ------------------------------------------------------------
# 5. Combine Dance rows only
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("RESULT")
print("=" * 100)

if len(results) == 0:
    print("No Dance major rows found.")
else:
    dance_df = pd.concat(results, ignore_index=True)

    print("Dance rows found:", len(dance_df))

    # Main count column
    value_col = count_col if count_col is not None else "record_count"

    print("\nCount column used:", value_col)

    # --------------------------------------------------------
    # Total Dance count
    # --------------------------------------------------------
    total_dance_count = dance_df[value_col].sum()

    print("\n" + "-" * 100)
    print("TOTAL DANCE MAJOR COUNT")
    print("-" * 100)
    print(f"{total_dance_count:,.0f}")

    # --------------------------------------------------------
    # Dance count by year
    # --------------------------------------------------------
    if year_col is not None:
        dance_by_year = (
            dance_df
            .groupby(year_col, dropna=False)[value_col]
            .sum()
            .reset_index()
            .sort_values(year_col)
        )

        print("\n" + "-" * 100)
        print("DANCE MAJOR COUNT BY YEAR")
        print("-" * 100)
        display(dance_by_year)

    # --------------------------------------------------------
    # Dance count by degree level
    # --------------------------------------------------------
    if degree_level_col is not None:
        dance_by_degree = (
            dance_df
            .groupby(degree_level_col, dropna=False)[value_col]
            .sum()
            .reset_index()
            .sort_values(value_col, ascending=False)
        )

        print("\n" + "-" * 100)
        print("DANCE MAJOR COUNT BY DEGREE LEVEL")
        print("-" * 100)
        display(dance_by_degree)

    # --------------------------------------------------------
    # Dance count by year + degree level
    # --------------------------------------------------------
    if year_col is not None and degree_level_col is not None:
        dance_by_year_degree = (
            dance_df
            .groupby([year_col, degree_level_col], dropna=False)[value_col]
            .sum()
            .reset_index()
            .sort_values([year_col, degree_level_col])
        )

        print("\n" + "-" * 100)
        print("DANCE MAJOR COUNT BY YEAR + DEGREE LEVEL")
        print("-" * 100)
        display(dance_by_year_degree)

    # --------------------------------------------------------
    # Show Dance rows preview
    # --------------------------------------------------------
    print("\n" + "-" * 100)
    print("DANCE ROWS PREVIEW")
    print("-" * 100)
    display(dance_df.head(20))

elapsed = (time.time() - start_time) / 60
print("\nDONE")
print(f"Total rows scanned: {total_rows_seen:,}")
print(f"Total time: {elapsed:.2f} minutes")

# Find Dance 2

In [ ]:
from pathlib import Path
import pandas as pd
import time
import gc

# ============================================================
# CLEAN DANCE MAJOR COUNT
# READ ONLY
# MEMORY OPTIMIZED
# NO SAVE
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

CHUNK_SIZE = 200_000

print("=" * 100)
print("CLEAN DANCE MAJOR COUNT")
print("=" * 100)
print("File:", DEGREE_FILE)

start_time = time.time()

USECOLS = [
    "year",
    "AWLEVEL",
    "CTOTALT",
    "cipcode",
    "CIPCODE",
    "cip_code",
    "major_name"
]

# Keep only columns that actually exist
all_columns = pd.read_csv(DEGREE_FILE, nrows=0, low_memory=False).columns.tolist()
USECOLS = [c for c in USECOLS if c in all_columns]

print("\nReading only columns:")
for c in USECOLS:
    print("-", c)

AWLEVEL_MAP = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate degree",
    4: "Certificate 2-4 years",
    5: "Bachelor degree",
    6: "Post-bachelor certificate",
    7: "Master degree",
    8: "Post-master certificate",
    9: "Doctor degree",
    17: "Doctor degree - other",
    18: "Doctor degree - professional/research",
    20: "Post-baccalaureate certificate",
    21: "Post-master certificate"
}

results = []
total_rows_seen = 0
dance_rows_found = 0
chunk_num = 0

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=USECOLS,
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
):
    chunk_num += 1
    total_rows_seen += len(chunk)

    # -----------------------------
    # Clean count
    # -----------------------------
    chunk["CTOTALT"] = (
        chunk["CTOTALT"]
        .fillna("0")
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
    )

    chunk["CTOTALT"] = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)

    # -----------------------------
    # Clean year
    # -----------------------------
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # -----------------------------
    # Clean AWLEVEL
    # Fix 5 and 5.0 into same value
    # -----------------------------
    chunk["AWLEVEL_CLEAN"] = pd.to_numeric(chunk["AWLEVEL"], errors="coerce").astype("Int64")

    # -----------------------------
    # Build Dance match
    # Dance CIP = 50.03xx
    # Example: 500301
    # -----------------------------
    dance_match = pd.Series(False, index=chunk.index)

    for c in ["cipcode", "CIPCODE", "cip_code"]:
        if c in chunk.columns:
            cip_clean = (
                chunk[c]
                .fillna("")
                .astype(str)
                .str.replace(".", "", regex=False)
                .str.replace("-", "", regex=False)
                .str.strip()
            )

            dance_match = dance_match | cip_clean.str.startswith("5003")

    if "major_name" in chunk.columns:
        dance_match = dance_match | chunk["major_name"].fillna("").str.contains(
            r"\bdance\b",
            case=False,
            regex=True
        )

    dance_chunk = chunk.loc[dance_match].copy()

    # Keep only real counts
    dance_chunk = dance_chunk[dance_chunk["CTOTALT"] > 0]

    if len(dance_chunk) > 0:
        dance_rows_found += len(dance_chunk)
        results.append(dance_chunk)

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {total_rows_seen:,} | "
            f"real dance rows found: {dance_rows_found:,} | total min: {elapsed:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

print("\n" + "=" * 100)
print("FINAL CLEAN RESULT")
print("=" * 100)

if len(results) == 0:
    print("No Dance rows with CTOTALT > 0 found.")
else:
    dance_df = pd.concat(results, ignore_index=True)

    dance_df["degree_level_name"] = dance_df["AWLEVEL_CLEAN"].map(AWLEVEL_MAP).fillna("Unknown / blank")

    total_dance_count = dance_df["CTOTALT"].sum()

    print("\nTOTAL REAL DANCE MAJOR COUNT:")
    print(f"{total_dance_count:,.0f}")

    # --------------------------------------------------------
    # Count by year
    # --------------------------------------------------------
    dance_by_year = (
        dance_df
        .groupby("year", dropna=False)["CTOTALT"]
        .sum()
        .reset_index()
        .sort_values("year")
    )

    print("\n" + "-" * 100)
    print("DANCE COUNT BY YEAR")
    print("-" * 100)
    display(dance_by_year)

    # --------------------------------------------------------
    # Count by degree level
    # --------------------------------------------------------
    dance_by_degree = (
        dance_df
        .groupby(["AWLEVEL_CLEAN", "degree_level_name"], dropna=False)["CTOTALT"]
        .sum()
        .reset_index()
        .sort_values("CTOTALT", ascending=False)
    )

    print("\n" + "-" * 100)
    print("DANCE COUNT BY DEGREE LEVEL")
    print("-" * 100)
    display(dance_by_degree)

    # --------------------------------------------------------
    # Count by year + degree level
    # --------------------------------------------------------
    dance_by_year_degree = (
        dance_df
        .groupby(["year", "AWLEVEL_CLEAN", "degree_level_name"], dropna=False)["CTOTALT"]
        .sum()
        .reset_index()
        .sort_values(["year", "AWLEVEL_CLEAN"])
    )

    print("\n" + "-" * 100)
    print("DANCE COUNT BY YEAR + DEGREE LEVEL")
    print("-" * 100)
    display(dance_by_year_degree)

    # --------------------------------------------------------
    # Preview real Dance rows
    # --------------------------------------------------------
    print("\n" + "-" * 100)
    print("REAL DANCE ROWS PREVIEW")
    print("-" * 100)
    display(dance_df.head(20))

elapsed = (time.time() - start_time) / 60

print("\nDONE")
print(f"Total rows scanned: {total_rows_seen:,}")
print(f"Total time: {elapsed:.2f} minutes")

# make a chart of it bar chart by year from beginning to end by count each year, and on top make it a how many in total

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time
import gc

# ============================================================
# DANCE MAJOR BAR CHART BY YEAR
# MEMORY OPTIMIZED
# READ SOURCE CSV
# SAVE CHART + TABLE TO DOWNLOADS
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
OUTPUT_DIR = Path.home() / "Downloads"
CHUNK_SIZE = 200_000

CHART_FILE = OUTPUT_DIR / "dance_major_count_by_year.png"
TABLE_FILE = OUTPUT_DIR / "dance_major_count_by_year.csv"

print("=" * 100)
print("MAKE DANCE BAR CHART BY YEAR")
print("=" * 100)
print("Source file:", DEGREE_FILE)
print("Chart output:", CHART_FILE)
print("Table output:", TABLE_FILE)

start_time = time.time()

# ------------------------------------------------------------
# Read header only first
# ------------------------------------------------------------
all_columns = pd.read_csv(DEGREE_FILE, nrows=0, low_memory=False).columns.tolist()

usecols = [c for c in ["year", "AWLEVEL", "CTOTALT", "cipcode", "CIPCODE", "cip_code", "major_name"] if c in all_columns]

print("\nReading only columns:")
for c in usecols:
    print("-", c)

# ------------------------------------------------------------
# Read in chunks
# ------------------------------------------------------------
results = []
chunk_num = 0
total_rows_seen = 0
dance_rows_found = 0

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
):
    chunk_num += 1
    total_rows_seen += len(chunk)

    # Clean count
    chunk["CTOTALT"] = (
        chunk["CTOTALT"]
        .fillna("0")
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
    )
    chunk["CTOTALT"] = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)

    # Clean year
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # Find Dance by CIP or major name
    dance_match = pd.Series(False, index=chunk.index)

    for c in ["cipcode", "CIPCODE", "cip_code"]:
        if c in chunk.columns:
            cip_clean = (
                chunk[c]
                .fillna("")
                .astype(str)
                .str.replace(".", "", regex=False)
                .str.replace("-", "", regex=False)
                .str.strip()
            )
            dance_match = dance_match | cip_clean.str.startswith("5003")

    if "major_name" in chunk.columns:
        dance_match = dance_match | chunk["major_name"].fillna("").str.contains(
            r"\bdance\b",
            case=False,
            regex=True
        )

    dance_chunk = chunk.loc[dance_match, ["year", "CTOTALT"]].copy()

    # keep only real counts
    dance_chunk = dance_chunk[dance_chunk["CTOTALT"] > 0]

    if len(dance_chunk) > 0:
        dance_rows_found += len(dance_chunk)
        results.append(dance_chunk)

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {total_rows_seen:,} | "
            f"dance rows kept: {dance_rows_found:,} | total min: {elapsed:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

# ------------------------------------------------------------
# Combine result
# ------------------------------------------------------------
if len(results) == 0:
    print("\nNo Dance rows found with CTOTALT > 0.")
else:
    dance_df = pd.concat(results, ignore_index=True)

    # Count by year
    dance_by_year = (
        dance_df
        .groupby("year", dropna=False)["CTOTALT"]
        .sum()
        .reset_index()
        .sort_values("year")
    )

    # optional: convert year to int if possible
    dance_by_year = dance_by_year[dance_by_year["year"].notna()].copy()
    dance_by_year["year"] = dance_by_year["year"].astype(int)

    total_dance = dance_by_year["CTOTALT"].sum()

    print("\n" + "=" * 100)
    print("FINAL RESULT")
    print("=" * 100)
    print(f"Total Dance count: {total_dance:,.0f}")
    print("\nDance by year:")
    display(dance_by_year)

    # Save table
    dance_by_year.to_csv(TABLE_FILE, index=False)

    # --------------------------------------------------------
    # Make bar chart
    # --------------------------------------------------------
    plt.figure(figsize=(16, 8))
    plt.bar(dance_by_year["year"].astype(str), dance_by_year["CTOTALT"])

    plt.title("Dance Major Count by Year", fontsize=14)
    plt.suptitle(f"Total Dance Count: {total_dance:,.0f}", fontsize=16, y=0.98)

    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save chart
    plt.savefig(CHART_FILE, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("\nSaved files:")
    print(CHART_FILE)
    print(TABLE_FILE)

elapsed = (time.time() - start_time) / 60
print(f"\nDONE | Total rows scanned: {total_rows_seen:,} | Total time: {elapsed:.2f} minutes")